# Step 5: From Zero-Shot Baseline to Fine-Tuning

This notebook follows a structured workflow:
1. **Zero-Shot Evaluation**: Replicating Step 4 performance by mapping COCO predictions to Cityscapes Train IDs.
2. **Fine-Tuning Setup**: Debugging the resolution and class-head mismatches to prepare the model for native Cityscapes training.

## 1. Environment Setup

In [1]:
!pip install lightning > /dev/null
!pip install gitignore_parser > /dev/null
!pip install -U 'jsonargparse[signatures]>=4.27.7' >/dev/null

In [2]:
from google.colab import drive
import os
import sys
import json
import yaml
import torch
import importlib
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from lightning import seed_everything

# 1. Mount Drive and Configure Paths
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

project_root = '/content/drive/MyDrive/FundGitHubProject'
eomt_folder = project_root + '/eomt'

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if eomt_folder not in sys.path:
    sys.path.insert(0, eomt_folder)

from eval.iouEval import iouEval
seed_everything(0, verbose=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

Active Device: cuda


## 2. Zero-Shot Baseline (The Step 4 Score)
Before fine-tuning, we verify that our model loading and mapping logic achieve the same results as Step 4.

In [5]:
from training.mask_classification_panoptic import MaskClassificationPanoptic
from models.eomt import EoMT
from models.vit import ViT

# 1. Load COCO Config and weights
coco_cfg_path = os.path.join(eomt_folder, 'configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml')
with open(coco_cfg_path, "r") as f: coco_config = yaml.safe_load(f)

coco_img_size = (640, 640)
bin_path = os.path.join(eomt_folder, 'eomt_weights/eomt_coco.bin')

# 2. Initialize Model (Standard COCO Configuration)
encoder = ViT(img_size=640, backbone_name="vit_base_patch14_reg4_dinov2")
network = EoMT(
    num_classes=133,
    encoder=encoder,
    num_q=200,
    num_blocks=3,
    masked_attn_enabled=False
)
model_coco = MaskClassificationPanoptic(
    network=network,
    img_size=(640, 640), # Changed from 640 to (640, 640)
    num_classes=133,
    stuff_classes=coco_config["data"].get("init_args", {}).get("stuff_classes", []),
    attn_mask_annealing_enabled=False
)

# Load weights
ckpt = torch.load(bin_path, map_location="cpu")
model_coco.load_state_dict(ckpt.get("state_dict", ckpt), strict=False)
model_coco.to(device).eval()
print("COCO Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


COCO Model loaded successfully!


In [ ]:
# 3. Setup Mapping (From Step 4)
map_file = os.path.join(project_root, 'coco-classes-mapping-master/coco_mapping_80to91.json')
with open(map_file, 'r') as f:
    coco_idx_map = {int(k)-1: int(v) for k, v in json.load(f).items()}

things_map = { 1: 11, 2: 18, 3: 13, 4: 17, 6: 15, 7: 16, 8: 14, 10: 6, 13: 7 }
stuff_map = { 100: 0, 123: 1, 91: 2, 129: 2, 109: 3, 110: 3, 111: 3, 112: 3, 131: 3, 117: 4, 116: 8, 125: 8, 126: 9, 119: 10 }

def bridge_to_cs(pred_tensor):
    res = torch.full_like(pred_tensor, 19)
    for idx, coco_id in coco_idx_map.items():
        if coco_id in things_map: res[pred_tensor == idx] = things_map[coco_id]
    for stuff_id, cs_id in stuff_map.items():
        res[pred_tensor == stuff_id] = cs_id
    return res

# 4. Initialize Cityscapes DataModule
from datasets.cityscapes_semantic import CityscapesSemantic
data_path = os.path.join(eomt_folder, 'data')
dm_cs = CityscapesSemantic(path=data_path, batch_size=1, num_workers=0)
dm_cs.setup()

# 5. Run Evaluation
evaluator = iouEval(20) # 19 classes + 1 ignore
for batch in tqdm(dm_cs.val_dataloader(), desc="Zero-Shot Baseline"):
    imgs, targets = batch
    gt = model_coco.to_per_pixel_targets_semantic(targets, 19)[0].to(device)

    with torch.no_grad():
        # Pre-process image (resize to 640 for COCO model)
        tx = model_coco.resize_and_pad_imgs_instance_panoptic([imgs[0].to(device)])
        mp, cp = model_coco(tx)
        mp = model_coco.revert_resize_and_pad_logits_instance_panoptic(
            F.interpolate(mp[-1], model_coco.img_size, mode="bilinear"),
            [imgs[0].shape[-2:]]
        )
        pred = model_coco.to_per_pixel_preds_panoptic(mp, cp[-1], model_coco.stuff_classes, 0.8, 0.8)[0][..., 0]

        # Map and Add to Evaluator
        evaluator.addBatch(bridge_to_cs(pred).unsqueeze(0).unsqueeze(0), gt.unsqueeze(0).unsqueeze(0))

_, ious = evaluator.getIoU()
print(f"\nZero-Shot Baseline mIoU: {ious[:19].mean()*100:.2f}%")

Zero-Shot Baseline:  26%|██▌       | 131/500 [01:24<03:38,  1.69it/s]

## 3. Preparing for Fine-Tuning

### The Resolution Challenge
The COCO pre-trained weights were trained at **640x640**. If we want to fine-tune on Cityscapes (often at 1024x1024 or similar), we must handle the **positional embeddings** mismatch in the ViT encoder.

In [ ]:
from training.mask_classification_semantic import MaskClassificationSemantic

# 1. Setup Target Configuration (Cityscapes Semantic)
num_classes = 19
target_img_size = (640, 640) # We stay at 640 for initial loading stability

encoder_ft = ViT(img_size=640, backbone_name="vit_base_patch14_reg4_dinov2")
network_ft = EoMT(
    num_classes=num_classes,
    encoder=encoder_ft,
    num_q=100, # Using Cityscapes default number of queries
    num_blocks=3,
    masked_attn_enabled=True
)

# 2. Initialize the Semantic Wrapper (The "Professor's" native class)
model_ft = MaskClassificationSemantic(
    network=network_ft,
    img_size=640,
    num_classes=num_classes,
    load_ckpt_class_head=False, # We don't want the 133-class head!
    ckpt_path=bin_path          # Let the library handle the surgery!
)

print("✅ Model surgically initialized for Fine-Tuning!")
print(f"Target classes: {model_ft.num_classes}")
print(f"Backbone loaded from: {bin_path}")

### Debugging the Input Size vs Model Size

If you try to run with `img_size=1024` while loading 640x640 weights, the `pos_embed` will mismatch. Here is how the codebase handles it under the hood (or how you can manually verify it):

In [ ]:
# Peek into the positional embeddings
ckpt_raw = torch.load(bin_path, map_location="cpu")
pos_embed_ckpt = ckpt_raw['state_dict']['network.encoder.backbone.pos_embed']
print(f"Checkpoint Pos Embed shape: {pos_embed_ckpt.shape}")
# Shape is [1, 1600, 768] -> 40x40 grid (640/16 = 40)

# If we wanted 1024, we would need 64x64 = 4096 tokens.
expected_tokens_1024 = (1024 // 16) ** 2
print(f"Tokens needed for 1024x1024: {expected_tokens_1024}")

# SUGGESTION: Stay at 640 for the start of fine-tuning to preserve the pre-trained spatial knowledge.
# If you must go higher, the library's 'ViT' class should be modified to interpolate during loading.

## 4. Next Step: Launch Training
Now that the model is loaded with pre-trained backbone/transformer weights and a fresh Cityscapes head, you can run:
```python
trainer.fit(model_ft, datamodule=data)
```

In [ ]:
trainer.fit(model_ft, datamodule=data)

## Now the peft library: